In [ ]:
import os
import sys
import json
from pathlib import Path
from urllib.parse import urljoin
import requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI

# Ensure `week1/scraper.py` is importable from this nested notebook folder.
for p in [Path.cwd(), *Path.cwd().parents]:
    scraper_dir = p / "week1"
    if (scraper_dir / "scraper.py").exists():
        sys.path.insert(0, str(scraper_dir))
        break

from scraper import fetch_website_contents, fetch_website_links

In [ ]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")

if google_api_key:
    print("API Key found and running")
else:
    print("API key not found")

MODEL = "gemini-2.5-flash-lite"
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

In [ ]:
links=fetch_website_links("https://pro.spyn.co/")
links

In [ ]:
link_system_prompt = """
You are given a list of links from a competitor website.
Read and Select links that are most relevant as an activity/academy center management platform
the content from which can be a part of a new AI-native SaaS platform brochure that provides the same services .

Return ONLY a JSON object in this exact shape:
{
  "links": [
    {"type": "about page", "url": "https://example.com/class/about"},
    {"type": "careers", "url": "https://example.com/class/career"}
  ]
}
"""

In [ ]:
def get_links_user_prompt(url):
    

    user_prompt = f"""
    I have shared a list of links from SpynPRO. 
    Decide the urls that will be helpful to make a company brochure for a competitor. 
    Respond with the full https URL in JSON format. 
    You can also quickly check the demo transcript, if possible to see what can be covered in the brochure
    Do not include Terms of Service, privacy policy pages, emails, release notes, and contact us pages.

Links (some might be relative links)
"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [ ]:
print(get_links_user_prompt("https://pro.spyn.co/"))

In [ ]:
def select_relevant_links(url):
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"},
        temperature=0
    )

    result = response.choices[0].message.content
    links = json.loads(result)

    normalized_links = []
    for link in links.get("links", []):
        if not isinstance(link, dict):
            continue
        absolute_url = urljoin(url, link.get("url", ""))
        if absolute_url.startswith(("http://", "https://")):
            normalized_links.append({**link, "url": absolute_url})

    links["links"] = normalized_links
    print(f"found {len(links['links'])} relevant links")
    return links


In [ ]:
select_relevant_links("https://pro.spyn.co/")

In [ ]:
def fetch_page_and_all_relevant_links(url):
    website_content = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page: \n\n{website_content}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        link_url = urljoin(url, link['url'])
        if not link_url.startswith(("http://", "https://")):
            continue
        result += f"\n\n## Link: {link['type']}\n"
        result += fetch_website_contents(link_url)
    return result


In [ ]:
print (fetch_page_and_all_relevant_links("https://pro.spyn.co/"))

In [ ]:
brochure_system_prompt = """ 
You are a marketing professional who is responsible to build a startup's brochure based on the competitors' links.
Analyze the content of the relevant links.
Respond in markdown without code blocks.
Include details about customers, competition and govenrment regulations.
"""

In [ ]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
    You are starting up a company: {company_name}.
    Here are the contents of its website and the relevant pages.
    Use this information to build a short brochure in markdown without code blocks. \n\n
    """
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000]
    return user_prompt

In [ ]:
get_brochure_user_prompt("SpynPRO", "https://pro.spyn.co/")

In [ ]:
def create_company_brochure(company_name, url):
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
       
    )
    result = response.choices[0].message.content
    display(Markdown(result)) 

In [ ]:
create_company_brochure("SpynPRO", "https://pro.spyn.co/")